In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_Silver_Folder_Permission"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Bronze/Folders_Inventory"
SOURCE_PATH = "abfss://Bronze/All_Permissions" # ← Change source path
TARGET_PATH = "abfss://silver/Dim_Folders_Permission" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_silver_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_silver_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_silver_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 3, Finished, Available, Finished)

🔧 Initializing ntk_Silver_Folder_Permission...
🚀 Starting ntk_Silver_Folder_Permission


In [2]:
source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Bronze_lakehouse.Lakehouse/Files/Bronze_layer/SharePointFiles"
target_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting"

StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 4, Finished, Available, Finished)

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

# Initialize Spark Session
spark = SparkSession.builder.appName("BronzeToSilver_ListsLibrary").getOrCreate()

today = datetime.now()  
from datetime import datetime, timedelta
today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d") 

bronze_path_folder = f"{source_path}/{year}/{month}/{day}/Folders_Inventory.csv"
bronze_path_prem = f"{source_path}/{year}/{month}/{day}/All_Permissions.csv"

current_date = datetime.now()
year = str(current_date.year)
month = f"{current_date.month:02d}"
day = f"{current_date.day:02d}"
silver_path = f"{target_path}/{year}/{month}/{day}/Dim_Folders_Permission.parquet"

# Load CSV with schema inference
df_source_folder = spark.read.option("header", "true").option("inferSchema", "true").option("multiline", "true").csv(bronze_path_folder)
df_source_folder.show(2)

df_source_prem = spark.read.option("header", "true").option("inferSchema", "true").option("multiline", "true").csv(bronze_path_prem)
df_source_prem.show(2)

StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 14, Finished, Available, Finished)

+-----------+--------------------+------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+----------------+-----------+------------+---------+--------------+--------------+---------------+------------+-------------------+-------+------+-------------+--------+--------------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------------+----------------+-----------------+-------+----------------+--------------+-------------+------------------+-------------------+-------------+------------------+---------------+-----------------+--------------------+--------------+------------+------------+
|LibraryName|          LibraryUrl|  FolderName|          FolderPath|           FolderUrl|ItemId|            UniqueId|                GUID|              SiteId|     SiteName|             S

In [13]:
from pyspark.sql.functions import col

# Assuming bronze_path_file is a DataFrame
df_source_folder = df_source_folder.drop("HasUniquePermissions")
# Show result
df_source_folder.show(3)

df_source_prem = df_source_prem.withColumn("PermissionLevel", split(trim(col("PermissionLevel")), ";"))
df_source_prem = df_source_prem.withColumn("PermissionLevel", explode(col("PermissionLevel")))
df_source_prem = df_source_prem.withColumn("PermissionLevel", trim(col("PermissionLevel")))

df_source_prem = df_source_prem.filter(col("ObjectType") == "Library")
df_source_prem.show(5)
print(f"✅ Total Library records: {df_source_prem.count()}")

StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 15, Finished, Available, Finished)

+-----------+--------------------+-------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+----------------+-----------+------------+---------+--------------+--------------+---------------+------------+-------------------+-------+------+-------------+--------+--------------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------------+----------------+-----------------+-------+----------------+--------------+-------------+------------------+-------------------+-------------+------------------+---------------+-----------------+--------------+------------+------------+
|LibraryName|          LibraryUrl|   FolderName|          FolderPath|           FolderUrl|ItemId|            UniqueId|                GUID|              SiteId|     SiteName|             SiteUrl|        Pare

In [14]:
df_cross_join = df_source_folder.crossJoin(df_source_prem)
df_cross_join.show(5)

print(df_cross_join.count())

StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 16, Finished, Available, Finished)

+-----------+--------------------+----------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+----------------+-----------+------------+---------+--------------+--------------+---------------+------------+-------------------+-------+------+-------------+--------+--------------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------------+----------------+-----------------+-------+----------------+--------------+-------------+------------------+-------------------+-------------+------------------+---------------+-----------------+--------------+------------+------------+--------------------+-----+----------+----------+--------------------+--------------------+------------+--------------------+--------------------+---------------+--------------------+----------------+-----------

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_cross = df_cross_join.select (
            "PermissionId",
            "Level",
            "FolderName",
            "ObjectName",
            "FolderUrl",
            "UniqueId",
            "ItemId",
            "ObjectCreated",
            "ObjectModified",
            "CreatedBy",
            "CreatedByEmail",
            "LastModifiedBy",
            "LastModifiedByEmail",
            "FolderSizeFormatted",
            "ObjectFileType",
            "ObjectFileExtension",
            "FolderPath",
            "ObjectContentType",
            "PrincipalType",
            "PrincipalName",
            "PrincipalEmail",
            "PrincipalLogin",
            "PrincipalId",
            "PrincipalUserPrincipalName",
            "IsGroup",
            "GroupId",
            "GroupType",
            "GroupOwner",
            "GroupMemberCount",
            "PermissionType",
            "PermissionLevel",
            "RoleDefinitionId",
            "PermissionScope",
            "PermissionGrantedDate",
            "PermissionGrantedBy",
            "PermissionModifiedDate",
            "PermissionModifiedBy",
            "InheritedFrom",
            "HasUniquePermissions",
            "InheritanceLevel",
            "ParentObjectUrl",
            "IsExternal",
            "IsSharedExternally",
            "DatacenterLocation",
            "SensitivityLabel",
            "Classification",
            "IsGuest",
            "IsSiteAdmin",
            "IsSiteOwner",
            "IsSystemAccount",
            "ReviewedDate",
            "ReviewedBy",
            "ReviewNotes",
            "ComplianceStatus",
            "RiskLevel",
            "RequiresReview",
            "CapturedDate",
            "CapturedBy",

)

df_cross.show(2)

StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 17, Finished, Available, Finished)

+--------------------+-----+----------+----------+--------------------+--------------------+------+--------------------+--------------------+--------------+--------------------+--------------+--------------------+-------------------+--------------+-------------------+--------------------+-----------------+---------------+--------------------+--------------+--------------------+-----------+--------------------------+-------+-------+---------------+--------------------+----------------+-----------------+---------------+--------------------+---------------+---------------------+-------------------+----------------------+--------------------+-------------+--------------------+----------------+--------------------+----------+------------------+------------------+----------------+--------------+-------+-----------+-----------+---------------+------------+----------+-----------+----------------+---------+--------------+--------------------+-------------+
|        PermissionId|Level|FolderName

In [16]:
from pyspark.sql.functions import concat_ws, sha2

df_cross = df_cross.withColumn(
    "HashKey",
    sha2(concat_ws("|", "FolderName", "FolderUrl", "FolderPath"), 256)
)
df_cross.show(5)

StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 18, Finished, Available, Finished)

+--------------------+-----+----------+----------+--------------------+--------------------+------+--------------------+--------------------+--------------+--------------------+--------------+--------------------+-------------------+--------------+-------------------+--------------------+-----------------+---------------+--------------------+--------------+--------------------+-----------+--------------------------+-------+-------+---------------+--------------------+----------------+-----------------+---------------+--------------------+---------------+---------------------+-------------------+----------------------+--------------------+-------------+--------------------+----------------+--------------------+----------+------------------+------------------+----------------+--------------+-------+-----------+-----------+---------------+------------+----------+-----------+----------------+---------+--------------+--------------------+-------------+--------------------+
|        Permissi

In [17]:
from pyspark.sql.functions import col, when, trim, sha2

# Step 1: Create identifier column with email as priority, fallback to display name
df_etl = df_cross.withColumn(
    "User_Ident",
    when(
        (col("PrincipalEmail").isNotNull()) & 
        (trim(col("PrincipalEmail")) != "") & 
        (col("PrincipalEmail") != " "),
        trim(col("PrincipalEmail"))
    ).otherwise(
        when(
            (col("PrincipalName").isNotNull()) & 
            (trim(col("PrincipalName")) != "") & 
            (col("PrincipalName") != " "),
            trim(col("PrincipalName"))
        ).otherwise(None)
    )
)

# Step 2: Create UserKey column (for PrincipalType == "User") using the updated df_etl
df_etl = df_etl.withColumn(
    "UserKey",
    when(
        (col("PrincipalType") == "User") & (col("User_Ident").isNotNull()),
        sha2(col("User_Ident"), 256)
    ).otherwise(None)
)

# Step 3: Create GroupKey column (for PrincipalType != "User")
df_etl = df_etl.withColumn(
    "GroupKey",
    when(
        (col("PrincipalType") != "User") & (col("User_Ident").isNotNull()),
        sha2(col("User_Ident"), 256)
    ).otherwise(None)
)

# Step 4: Create ObjectKey hash value (using FolderUrl)
df_etl = df_etl.withColumn(
    "ObjectKey",
    when(
        col("FolderUrl").isNotNull(),
        sha2(col("FolderUrl"), 256)
    ).otherwise(None)
)

# Step 5: Create PermissionKey hash value (using PermissionLevel)
df_etl = df_etl.withColumn(
    "PermissionKey",
    when(
        col("PermissionLevel").isNotNull(),
        sha2(col("PermissionLevel"), 256)
    ).otherwise(None)
)

df_etl.show(5)
print("UserKey, GroupKey, ObjectKey, and PermissionKey created successfully.")


StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 19, Finished, Available, Finished)

+--------------------+-----+----------+----------+--------------------+--------------------+------+--------------------+--------------------+--------------+--------------------+--------------+--------------------+-------------------+--------------+-------------------+--------------------+-----------------+---------------+--------------------+--------------+--------------------+-----------+--------------------------+-------+-------+---------------+--------------------+----------------+-----------------+---------------+--------------------+---------------+---------------------+-------------------+----------------------+--------------------+-------------+--------------------+----------------+--------------------+----------+------------------+------------------+----------------+--------------+-------+-----------+-----------+---------------+------------+----------+-----------+----------------+---------+--------------+--------------------+-------------+--------------------+------------------

In [18]:
### Write into Bronz layer    5a30e3d23bac75e9e
df_etl.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(silver_path)

# df_cleaned.printSchema()
print(f"Dim_Folders_Permission.Parquet file created successfully in Silver layer! {silver_path}")


StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 20, Finished, Available, Finished)

Dim_Folders_Permission.Parquet file created successfully in Silver layer! abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_Folders_Permission.parquet


In [19]:
df_etl.printSchema()

StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 21, Finished, Available, Finished)

root
 |-- PermissionId: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- FolderName: string (nullable = true)
 |-- ObjectName: string (nullable = true)
 |-- FolderUrl: string (nullable = true)
 |-- UniqueId: string (nullable = true)
 |-- ItemId: integer (nullable = true)
 |-- ObjectCreated: string (nullable = true)
 |-- ObjectModified: string (nullable = true)
 |-- CreatedBy: string (nullable = true)
 |-- CreatedByEmail: string (nullable = true)
 |-- LastModifiedBy: string (nullable = true)
 |-- LastModifiedByEmail: string (nullable = true)
 |-- FolderSizeFormatted: string (nullable = true)
 |-- ObjectFileType: string (nullable = true)
 |-- ObjectFileExtension: string (nullable = true)
 |-- FolderPath: string (nullable = true)
 |-- ObjectContentType: string (nullable = true)
 |-- PrincipalType: string (nullable = true)
 |-- PrincipalName: string (nullable = true)
 |-- PrincipalEmail: string (nullable = true)
 |-- PrincipalLogin: string (nullable = true)
 |-- Principal

In [11]:
from datetime import datetime

# Define the log_etl_activity function for logging ETL process
def log_etl_activity(status, start_time, rows_read=None, rows_written=None, bytes_processed=None, error_details=None):

    end_time = datetime.now()
    duration_seconds = (end_time - start_time).total_seconds()

    log_message = {
        'Status': status,
        'StartTime': start_time,
        'EndTime': end_time,
        'DurationSeconds': duration_seconds,
        'RowsRead': rows_read,
        'RowsWritten': rows_written,
        'BytesProcessed': bytes_processed,
        'ErrorDetails': error_details
    }

    # For simplicity, let's print the log message (this can be replaced with a logging system)
    print("Logging ETL Activity:", log_message)

# Ensure processing_successful is defined before this block
try:
    NOTEBOOK_NAME = "ETL_Pipeline_Example"  # Define your notebook name or use the existing one
    start_time = datetime.now()  # Capture the start time of the ETL process

    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Simulated metrics
    rows_read = df_source_folder.count()   # Correct this to have a meaningful `rows_read`
    rows_read = df_source_prem.count()
    rows_written = df_etl.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details=error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, 0cad1dd0-dc5c-4f14-bde8-83d19ed0f7a9, 13, Finished, Available, Finished)

🔄 Starting ETL processing for ETL_Pipeline_Example...
Logging ETL Activity: {'Status': 'SUCCESS', 'StartTime': datetime.datetime(2025, 10, 15, 4, 48, 6, 893156), 'EndTime': datetime.datetime(2025, 10, 15, 4, 48, 7, 515557), 'DurationSeconds': 0.622401, 'RowsRead': 19, 'RowsWritten': 5909, 'BytesProcessed': 524288000, 'ErrorDetails': None}
🎉 ETL_Pipeline_Example pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 19 → 5,909
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ETL_Pipeline_Example:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ETL_Pipeline_Example logging completed!
